# Aula 11: Fundamentos de Tabelas Hash e Endereçamento Aberto

**Objetivo:** Compreender o que é uma Tabela Hash, o papel da função de hashing, o problema das colisões e aprender a implementar uma Tabela Hash usando a estratégia de resolução de colisões por **Endereçamento Aberto** (Sondagem Linear).

### **Parte 1: Conceitos Fundamentais de Hashing**

#### **1. O que é uma Tabela Hash?**

Até agora, vimos estruturas que acessam dados por um índice numérico (arrays) ou por travessia (listas, árvores). Uma **Tabela Hash** (ou *Hash Table*) é uma estrutura de dados que implementa um *array associativo*, permitindo o armazenamento e acesso a dados através de um par **{chave-valor}**. 

Pense em um dicionário de contatos: em vez de procurar alguém pelo índice `56`, você procura pela chave `"Maria"`. Tabelas Hash são projetadas para tornar essa busca extremamente rápida, idealmente em tempo constante **O(1)**.

#### **2. A Função de Hashing**

A "mágica" por trás da Tabela Hash é a **função de hashing**. Seu trabalho é converter uma chave de qualquer tipo (como uma string) em um número inteiro, que será usado como o índice do array onde o valor correspondente será armazenado.

* **Exemplo Simples (e falho):** Somar os valores ordinais (Unicode/ASCII) dos caracteres de uma string.

Ver figura 8.2

In [3]:
ord
# 'a' -> 97, 'd' -> 100
sum(map(ord, 'ad')) # Retorna 197
    
# 'g' -> 103, 'a' -> 97
sum(map(ord, 'ga')) # Retorna 200 - Opa, diferente! 
# Correção do livro: 'ga' seria 103+97=200, ad seria 97+100=197. O livro mostra que 'ad' e 'ga' colidem com uma função hash imperfeita, mas o exemplo de soma simples não causa isso. O exemplo de colisão do livro (p. 252) é 'ad' e 'ga' com a função myhash.

# Mas...
sum(map(ord, 'world hello')) # Retorna 1116
sum(map(ord, 'hello world')) # Retorna 1116

1116

Este método simples gera o mesmo hash para anagramas, o que é um problema.

* **Uma Função de Hashing Melhor:** Para evitar colisões simples, podemos usar um multiplicador que muda a cada caractere.

In [4]:
    def minha_hash(chave_str):
        mult = 1
        valor_hash = 0
        for char in chave_str:
            valor_hash += mult * ord(char)
            mult += 1
        return valor_hash
    
    print(f"'hello world': {minha_hash('hello world')}") # 6736
    print(f"'world hello': {minha_hash('world hello')}") # 6616 (Diferente!)

'hello world': 6736
'world hello': 6616


#### **3. O Problema da Colisão**

Mesmo com uma função melhor, é inevitável que duas chaves diferentes possam gerar o mesmo índice de hash. Isso é chamado de **colisão**.

`minha_hash('ad') % 256` e `minha_hash('ga') % 256` podem resultar no mesmo índice. Uma Tabela Hash funcional *precisa* de uma estratégia para lidar com colisões.

### **Parte 2: Resolução de Colisões com Endereçamento Aberto**

**Endereçamento Aberto** (*Open Addressing*) é uma estratégia onde, em caso de colisão, procuramos por outro slot (ou "bucket") vazio **dentro da própria tabela** para inserir o novo item. 

#### **1. Sondagem Linear (Linear Probing)**

A forma mais simples de endereçamento aberto é a **sondagem linear**. Se o slot `h` está ocupado, tentamos `h+1`, depois `h+2`, `h+3`, e assim por diante, até encontrar um espaço livre. 

* **Vantagem:** Simples de implementar.
* **Desvantagem:** Causa **clustering primário**, que são aglomerados de dados na tabela, o que pode degradar a performance, tornando as buscas e inserções mais lentas. 

#### **2. Implementação da Tabela Hash com Sondagem Linear**

Vamos construir nossa classe `TabelaHash`.

In [ ]:
class ItemHash:
    """Armazena o par {chave-valor}."""
    def __init__(self, chave, valor):
        self.chave = chave
        self.valor = valor

class TabelaHash:
    def __init__(self):
        self.tamanho = 256
        self.slots = [None] * self.tamanho
        self.contador = 0 # Itens inseridos

    def _hash(self, chave):
        """Nossa função de hashing para calcular o índice."""
        mult = 1
        valor_hash = 0
        for char in chave:
            valor_hash += mult * ord(char)
            mult += 1
        return valor_hash % self.tamanho

    def put(self, chave, valor):
        """Insere um item na tabela."""
        item = ItemHash(chave, valor)
        h = self._hash(chave)

        # Sondagem Linear para encontrar um slot
        while self.slots[h] is not None:
            if self.slots[h].chave == chave:
                break # Se a chave já existe, vamos sobrescrever
            h = (h + 1) % self.tamanho
        
        if self.slots[h] is None:
            self.contador += 1
        self.slots[h] = item

    def get(self, chave):
        """Recupera um valor da tabela."""
        h = self._hash(chave)

        # Sondagem Linear para encontrar a chave
        while self.slots[h] is not None:
            if self.slots[h].chave == chave:
                return self.slots[h].valor
            h = (h + 1) % self.tamanho
        
        return None # Chave não encontrada

### **Parte 3: Melhorando a Tabela Hash **

#### **1. Crescimento da Tabela e Fator de Carga**

À medida que inserimos itens, a tabela enche, e as colisões se tornam mais frequentes. Para manter a performance, precisamos aumentar o tamanho da tabela quando ela atinge um certo nível de ocupação.

  * **Fator de Carga (Load Factor):** É a métrica que usamos para decidir quando crescer a tabela.
      * `Fator de Carga = itens_inseridos / tamanho_total_da_tabela`
  * Normalmente, definimos um limite (ex: 0.75 ou 75%). Quando o fator de carga ultrapassa esse limite, dobramos o tamanho da tabela e fazemos **rehashing** de todos os itens existentes.

In [ ]:
# Adicionar estes métodos na classe TabelaHash

def _crescer_tabela(self):
    """Dobra o tamanho da tabela e faz o rehashing de todos os itens."""
    tabela_antiga = self.slots
    self.tamanho *= 2
    self.slots = [None] * self.tamanho
    self.contador = 0
    
    print("Crescendo tabela! Novo tamanho:", self.tamanho)
    
    for item in tabela_antiga:
        if item is not None:
            self.put(item.chave, item.valor)

# Precisamos modificar o método put() para chamar esta verificação
def put(self, chave, valor):
    # (código anterior do put)
    # ...
    if self.slots[h] is None:
        self.contador += 1
    self.slots[h] = item

    # Verifica o fator de carga após a inserção
    if (self.contador / self.tamanho) > 0.75:
        self._crescer_tabela()

#### **2. Outras Técnicas de Sondagem (Visão Geral)**

Para mitigar o problema do *clustering primário* da sondagem linear, existem outras técnicas de endereçamento aberto:

  * **Sondagem Quadrática (Quadratic Probing):** Em vez de tentar `h+1, h+2, ...`, tenta-se `h+1²`, `h+2²`, `h+3²`, ... Isso ajuda a espalhar mais os itens, mas pode criar seu próprio padrão de *clustering secundário*. 
  * **Hashing Duplo (Double Hashing):** Usa uma segunda função de hash para determinar o tamanho do "salto" a cada tentativa. `(h1(chave) + i * h2(chave)) % tamanho`. Esta é a técnica de endereçamento aberto mais eficaz para evitar clustering. 

### **Exercícios da Aula 11**

1.  Existe uma tabela hash com 400 slots e 200 elementos armazenados nela. Qual será o fator de carga da tabela hash? 
2.  Descreva o problema conhecido como "clustering primário" que ocorre com a sondagem linear. Por que a sondagem quadrática ou o hashing duplo são considerados melhores a esse respeito?
3.  Implemente os métodos especiais `__putitem__(self, chave, valor)` e `__getitem__(self, chave)` na classe `TabelaHash` para que ela possa ser usada com a sintaxe de dicionário (ex: `tabela['minha_chave'] = 'meu_valor'`).

